# Fidelis 2.0: natural image restoration

This notebook runs VOSR 2.0 on a free interactive Colab GPU. Use **Runtime > Run all**. When prompted, select one or more source images. The finished images download as one ZIP.

The first run downloads about 7 GB of model data into the temporary Colab session. It does not use your Google Drive.

In [ ]:
# @title 1. Prepare the VOSR 2.0 GPU environment
import os, shutil, subprocess, sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError('No Colab GPU is active. Choose Runtime > Change runtime type > T4 GPU, then run all again.')

gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 1024**3
if gpu_gb < 14:
    raise RuntimeError(f'{gpu.name} has only {gpu_gb:.1f} GB VRAM. Fidelis requires a T4-class GPU with at least 14 GB.')
print(f'GPU: {gpu.name}, {gpu_gb:.1f} GB')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

ROOT = Path('/content/fidelis-v2')
COMFY = ROOT / 'ComfyUI'
NODE = COMFY / 'custom_nodes' / 'ComfyUI-VOSR2'
COMFY_COMMIT = 'd43a5fa20c8547ff42d13232f589a06536c42b97'
NODE_COMMIT = 'c9450b611e6b0e854212b81fecde2da5d088c1bc'
VOSR_REFERENCE_COMMIT = '516f292b99cf23c76fdc33351e86dc4f97711fe8'

ROOT.mkdir(parents=True, exist_ok=True)
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY)], check=True)
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth=1', 'origin', COMFY_COMMIT], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--detach', COMFY_COMMIT], check=True)

NODE.parent.mkdir(parents=True, exist_ok=True)
if not NODE.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/ylchen333/ComfyUI-VOSR2.git', str(NODE)], check=True)
subprocess.run(['git', '-C', str(NODE), 'fetch', '--depth=1', 'origin', NODE_COMMIT], check=True)
subprocess.run(['git', '-C', str(NODE), 'checkout', '--detach', NODE_COMMIT], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.34,<1'], check=True)
print('Environment ready. VOSR reference:', VOSR_REFERENCE_COMMIT)

In [ ]:
# @title 2. Select the original images
from google.colab import files

INPUT_DIR = ROOT / 'input'
OUTPUT_DIR = ROOT / 'output'
shutil.rmtree(INPUT_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True)

uploaded = files.upload()
allowed = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
saved = []
for original_name, data in uploaded.items():
    name = Path(original_name).name
    if Path(name).suffix.lower() not in allowed:
        print('Skipped unsupported file:', name)
        continue
    target = INPUT_DIR / name
    target.write_bytes(data)
    saved.append(target)
if not saved:
    raise RuntimeError('No supported image was selected.')
print(f'Ready to restore {len(saved)} image(s).')

In [ ]:
# @title 3. Restore at 4x and download
import gc, importlib, types, zipfile
import numpy as np
from PIL import Image

os.chdir(COMFY)
sys.argv = [sys.argv[0], '--lowvram']
sys.path.insert(0, str(COMFY))
package_name = 'fidelis_vosr2'
package = types.ModuleType(package_name)
package.__path__ = [str(NODE)]
sys.modules[package_name] = package
loader = importlib.import_module(f'{package_name}.loader')
inference = importlib.import_module(f'{package_name}.inference')

print('Downloading and loading VOSR 2.0. The first run downloads about 7 GB...')
model = loader.load_vosr2('VOSR2', 'fp16')

completed = []
for index, source in enumerate(saved, 1):
    print(f'[{index}/{len(saved)}] Restoring {source.name}')
    with Image.open(source) as opened:
        rgb = opened.convert('RGB')
        image = torch.from_numpy(np.asarray(rgb).copy()).float().div(255).unsqueeze(0)
    with torch.inference_mode():
        result = inference.run_vosr2(
            model=model, images_bhwc01=image, upscale=4, seed=42,
            color_alignment='wavelet', tile_size=512, tile_overlap=64,
            vae_tile_size=1024, vae_tile_overlap=128,
        )
    pixels = result[0].clamp(0, 1).mul(255).round().byte().cpu().numpy()
    destination = OUTPUT_DIR / f'{source.stem}-fidelis-4x.png'
    Image.fromarray(pixels).save(destination, format='PNG', optimize=True)
    completed.append(destination)
    del image, result, pixels
    gc.collect()
    torch.cuda.empty_cache()

archive = ROOT / 'fidelis-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for output in completed:
        bundle.write(output, arcname=output.name)
print(f'Finished {len(completed)} image(s). Downloading {archive.name}...')
files.download(str(archive))